In [7]:
import pandas as pd
import numpy as np
from google.colab import files

# 1. 파일 업로드
files.upload()

# 2. 데이터 불러오기 (df 변수 생성)
DATA = 'knhanes_like_week4.csv'
df = pd.read_csv(DATA)

print(f"데이터 로드 완료! 전체 샘플 수: {len(df)}개")

Saving knhanes_like_week4.csv to knhanes_like_week4 (1).csv
데이터 로드 완료! 전체 샘플 수: 5367개


In [8]:
import pandas as pd

# 데이터 불러와서 df 변수에 저장
df = pd.read_csv('knhanes_like_week4.csv')
print("df 변수가 정상적으로 생성되었습니다!")

df 변수가 정상적으로 생성되었습니다!


In [9]:
import numpy as np
import pandas as pd
from scipy import stats

# ==============================================================================
# 0. 데이터 로드 (df 생성)
# ==============================================================================
try:
    df = pd.read_csv('knhanes_like_week4.csv')
    print("✅ 데이터(df) 로드 성공!")
except Exception as e:
    print("❌ 'knhanes_like_week4.csv' 파일을 찾을 수 없습니다. 파일 경로나 업로드 상태를 확인해 주세요.")

# ==============================================================================
# 1. 필수 계산 함수 정의 (선행 셀 미실행 문제 방지)
# ==============================================================================
def _var_psu(z, strata, psu):
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var

def svymean_domain(y, w, strata, psu, domain):
    y = np.asarray(y, float)
    d = np.asarray(domain, float) * (~np.isnan(y))
    y = np.nan_to_num(y) * d
    wd = np.asarray(w, float) * d

    sum_wd = np.sum(wd)
    if sum_wd == 0:
        return np.nan, np.nan

    th = np.sum(wd * y) / sum_wd
    return th, np.sqrt(_var_psu(wd * (y - th) / sum_wd, strata, psu))

# ==============================================================================
# 2. 데이터 전처리 및 무응답 코드(9) 결측 처리 (피드백 반영)
# ==============================================================================
df_clean = df.copy()

for col in ['income', 'edu', 'smk', 'drink', 'exercise']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(9, np.nan)

# 가중치(wt), 층(kstrata), 집락(psu) 설정
w = df_clean['wt'].values
ST = df_clean['kstrata'].values
PS = df_clean['psu'].values

# ==============================================================================
# 3. 도메인 설정 (데이터 잘라내기 금지 지침 준수)
# ==============================================================================
dong = (df_clean.urban == 1).values
eup = (df_clean.urban == 2).values

DOMS = {
    '전체 (N=5,367)': np.ones(len(df_clean), bool),
    '동부 (N=3,211)': dong,
    '읍면부 (N=2,156)': eup
}

# ==============================================================================
# 4. 연속형 & 범주형 행 생성 함수 (p-value, SMD 자동 계산)
# ==============================================================================
def row_cont(label, var):
    y = df_clean[var].values
    valid_n = int(pd.notna(df_clean[var]).sum())

    m1, se1 = svymean_domain(y, w, ST, PS, dong)
    m2, se2 = svymean_domain(y, w, ST, PS, eup)

    W1 = np.sum(w * dong * pd.notna(y))
    W2 = np.sum(w * eup * pd.notna(y))
    yy = np.nan_to_num(y)
    z_diff = (w * dong * (yy - m1)) / W1 - (w * eup * (yy - m2)) / W2
    se_diff = np.sqrt(_var_psu(z_diff, ST, PS))
    t_stat = (m1 - m2) / se_diff if se_diff > 0 else 0
    p_val = 2 * (1 - stats.norm.cdf(abs(t_stat)))

    s1 = np.sqrt(np.sum(w[dong] * (y[dong] - m1)**2) / np.sum(w[dong])) if np.sum(w[dong]) > 0 else 0
    s2 = np.sqrt(np.sum(w[eup] * (y[eup] - m2)**2) / np.sum(w[eup])) if np.sum(w[eup]) > 0 else 0
    smd = abs(m1 - m2) / np.sqrt((s1**2 + s2**2) / 2) if (s1 > 0 or s2 > 0) else 0.0

    r = {'특성': label, 'n': valid_n}
    for name, dom in DOMS.items():
        th, se = svymean_domain(y, w, ST, PS, dom)
        r[name] = f'{th:.1f} ({se:.2f})'
    r['p-value'] = f'{p_val:.3f}' if p_val >= 0.001 else '< 0.001'
    r['SMD'] = f'{smd:.2f}'
    return r

def row_cat(label, mask):
    mask_arr = np.asarray(mask, float)
    valid_n = int(np.nansum(mask_arr == 1))

    p1, _ = svymean_domain(mask_arr, w, ST, PS, dong)
    p2, _ = svymean_domain(mask_arr, w, ST, PS, eup)

    q1, q2 = 1 - p1, 1 - p2
    smd = abs(p1 - p2) / np.sqrt((p1*q1 + p2*q2) / 2) if (p1 > 0 or p2 > 0) else 0.0

    W1, W2 = np.sum(w * dong), np.sum(w * eup)
    z_diff = (w * dong * (mask_arr - p1)) / W1 - (w * eup * (mask_arr - p2)) / W2
    se_diff = np.sqrt(_var_psu(z_diff, ST, PS))
    z_stat = (p1 - p2) / se_diff if se_diff > 0 else 0
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    r = {'특성': label, 'n': valid_n}
    for name, dom in DOMS.items():
        th, _ = svymean_domain(mask_arr, w, ST, PS, dom)
        dom_n = int(np.nansum((mask_arr == 1) & dom))
        r[name] = f'{dom_n:,} ({th*100:.1f}%)'
    r['p-value'] = f'{p_val:.3f}' if p_val >= 0.001 else '< 0.001'
    r['SMD'] = f'{smd:.2f}'
    return r

# ==============================================================================
# 5. Table 1 최종 생성
# ==============================================================================
t1_final = pd.DataFrame([
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df_clean.age >= 65),
    row_cat('성별 (남성)', df_clean.sex == 1),
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cat('  비만 (BMI ≥ 25)', df_clean.bmi >= 25),
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('  고혈압 유병', df_clean.htn == 1),
    row_cont('공복혈당 (mg/dL)', 'glucose'),
    row_cat('  당뇨병 (혈당 ≥ 126)', df_clean.glucose >= 126),
    row_cont('총콜레스테롤 (mg/dL)', 'chol'),
    row_cat('현재 흡연', df_clean.smk == 1),
    row_cat('규칙적 운동', df_clean.exercise == 2),
    row_cat('고위험 음주', df_clean.drink == 3)
])

t1_final

✅ 데이터(df) 로드 성공!


,특성,n,"전체 (N=5,367)","동부 (N=3,211)","읍면부 (N=2,156)",p-value,SMD
0,연령 (세),5367,49.1 (0.16),47.2 (0.20),55.8 (0.23),< 0.001,0.56
1,65세 이상,1607,"1,607 (16.8%)",741 (14.0%),866 (26.2%),< 0.001,0.31
2,성별 (남성),2719,"2,719 (49.0%)","1,611 (49.0%)","1,108 (49.1%)",0.943,0.00
3,BMI (kg/m²),5367,23.8 (0.05),23.7 (0.07),24.2 (0.08),< 0.001,0.12
4,비만 (BMI ≥ 25),2035,"2,035 (36.5%)","1,164 (35.6%)",871 (39.5%),0.004,0.08
5,수축기혈압 (mmHg),5367,122.8 (0.45),121.4 (0.53),127.3 (0.75),< 0.001,0.33
6,고혈압 유병,1652,"1,652 (24.4%)",838 (22.0%),814 (32.3%),< 0.001,0.23
7,공복혈당 (mg/dL),5000,91.6 (0.25),91.2 (0.31),93.0 (0.31),0.036,0.00
8,당뇨병 (혈당 ≥ 126),146,146 (2.1%),76 (2.1%),70 (2.5%),0.288,0.03
9,총콜레스테롤 (mg/dL),4806,170.4 (0.49),169.5 (0.58),173.5 (0.81),0.055,0.00
